In [1]:
import os
import importlib.machinery
import json

## Global Variables

In [2]:
# specify input variable file
in_GlobalVar_txt     = "py_Variables - vt_JsonVars.txt"

# get path to script launch point (i.e. location of 'HailMary.s' or '*.ipynb')
dir_ScriptLaunch = os.getcwd()

path_in_GlobalVar_txt = os.path.join(dir_ScriptLaunch, in_GlobalVar_txt)

# create variables from input file global variables
GlobalVars = importlib.machinery.SourceFileLoader('data', path_in_GlobalVar_txt).load_module()

ParentDir         = GlobalVars.ParentDir
ScenarioDir       = GlobalVars.ScenarioDir
ModelVersion      = GlobalVars.ModelVersion
ScenarioGroup     = GlobalVars.ScenarioGroup
RunYear           = int(GlobalVars.RunYear)
output_path       = os.path.join(ParentDir, r"Scenarios\.vizTool\config")

In [4]:
# read viztool config file and get scenario variables
with open(os.path.join(dir_ScriptLaunch,"../config.json")) as file:
    config_data = json.load(file)

file_name = config_data.get("scenarios", {}).get("file_name")
start_new = config_data.get("scenarios", {}).get("start_new")
initial_select         = config_data.get("scenarios", {}).get("initial_select")
initial_select_compare = config_data.get("scenarios", {}).get("initial_select_compare")
trendless = config_data.get("scenarios", {}).get("trendless")

## Functions

In [5]:
def append_update(data, jsonGroupName, jsonVarName, jsonGroup):
    updated=False
    for existing in data[jsonGroupName]:
        if existing[jsonVarName] == jsonGroup[jsonVarName]:
            existing.update(jsonGroup)
            updated = True
            break
    
    if not updated:
        data[jsonGroupName].append(jsonGroup)
    
def update_scenarios(data, trendless):
    for scenario in data['scenarios']:
        if scenario['scnGroup'] in trendless:
            # Get all scnGroups for the modVersion
            modVersion = scenario['modVersion']
            scnTrendCodes = list(set(
                [scn['scnGroup']
                for scn in data['scenarios']
                if scn['modVersion'] == modVersion and scn['scnGroup'] not in trendless]
            ))
            # Update the scnTrendCodes
            scenario['scnTrendCodes'] = scnTrendCodes
        else:
            scenario['scnTrendCodes'] = [scenario['scnTrendCodes']] if not isinstance(scenario['scnTrendCodes'], list) else scenario['scnTrendCodes']

## Compile Data for Json

In [6]:
scenarios_json_path = os.path.join(output_path, file_name)


if ((os.path.exists(scenarios_json_path)) & (start_new==False)):
    # Read the existing JSON file
    with open(scenarios_json_path, 'r') as file:
        data = json.load(file)
else:
    # Initialize data if the file does not exist
    data = {
        "initial_select": [],
        "initial_select_compare": [],
        "models": [],
        "trends": [],
        "scenarios": []
    }

In [7]:
if initial_select == "":
    is_modVersion, is_scnGroup, is_scnYear = ModelVersion, ScenarioGroup, RunYear
else:
    is_modVersion, is_scnGroup, is_scnYear = initial_select.split("__")
    is_scnYear = int(is_scnYear)

if initial_select_compare == "":
    isc_modVersion, isc_scnGroup, isc_scnYear = ModelVersion, ScenarioGroup, RunYear
else:
    isc_modVersion, isc_scnGroup, isc_scnYear = initial_select_compare.split("__")
    isc_scnYear = int(isc_scnYear)

In [8]:
geojsons = {
    "taz"    : "taz_"                 + ModelVersion + ".geojson",
    "segment": "segments_"            + ModelVersion + ".geojson",
    "city"   : "cities_"              + ModelVersion + ".geojson",
    "distSml": "districtsSmall_"      + ModelVersion + ".geojson",
    "distMed": "districtsMedium_"     + ModelVersion + ".geojson",
    "distLrg": "districtsLarge_"      + ModelVersion + ".geojson",
    "distSup": "districtsSuper_"      + ModelVersion + ".geojson",
    "subarea": "subareaWasatchFront_" + ModelVersion + ".geojson"
}

In [9]:
data["initial_select"] = [{
    "modVersion": is_modVersion, 
    "scnGroup": is_scnGroup, 
    "scnYear": is_scnYear
}]

data["initial_select_compare"] = [{
    "modVersion": isc_modVersion, 
    "scnGroup": isc_scnGroup, 
    "scnYear": isc_scnYear
}]

In [10]:
model = {
    "modVersion": ModelVersion, 
    "geojsons": geojsons
}

append_update(data, "models", "modVersion", model)

In [11]:
if ScenarioGroup in trendless:
    trend = {}
else:
    trend = {
        "scnTrendCode": ScenarioGroup, 
        "displayName": ScenarioGroup, 
        "displayByDefault": True
    }
    append_update(data, "trends", "scnTrendCode", trend)

In [12]:
scnCode = ModelVersion +'__'+ ScenarioGroup +'__'+  str(RunYear)

scenarios = {
    "modVersion": ModelVersion,
    "scnGroup": ScenarioGroup,
    "scnYear": RunYear,
    "scnCode": scnCode,
    "alias": ScenarioGroup + ' ' + str(RunYear),
    "scnTrendCodes": ScenarioGroup
}

append_update(data, "scenarios", "scnCode", scenarios)
update_scenarios(data, trendless)

In [13]:
data

{'initial_select': [{'modVersion': 'v910',
   'scnGroup': 'Base',
   'scnYear': 2019}],
 'initial_select_compare': [{'modVersion': 'v910',
   'scnGroup': 'Base',
   'scnYear': 2019}],
 'models': [{'modVersion': 'v910',
   'geojsons': {'taz': 'taz_v910.geojson',
    'segment': 'segments_v910.geojson',
    'city': 'cities_v910.geojson',
    'distSml': 'districtsSmall_v910.geojson',
    'distMed': 'districtsMedium_v910.geojson',
    'distLrg': 'districtsLarge_v910.geojson',
    'distSup': 'districtsSuper_v910.geojson',
    'subarea': 'subareaWasatchFront_v910.geojson'}}],
 'trends': [{'scnTrendCode': 'RTP',
   'displayName': 'RTP',
   'displayByDefault': True}],
 'scenarios': [{'modVersion': 'v910',
   'scnGroup': 'Base',
   'scnYear': 2019,
   'scnCode': 'v910__Base__2019',
   'alias': 'Base 2019',
   'scnTrendCodes': ['RTP']},
  {'modVersion': 'v910',
   'scnGroup': 'RTP',
   'scnYear': 2032,
   'scnCode': 'v910__RTP__2032',
   'alias': 'RTP 2032',
   'scnTrendCodes': ['RTP']}]}

In [14]:
json_data = json.dumps(data, indent=4)

with open(scenarios_json_path, 'w') as file:
    file.write(json_data)